# L5 · Notebook 03 — SGD 是 RM 的特例

**对应 PPT**：`L5-SA.tex` ★ SGD 算法 = RM 的特例（line 465）

## 教学目标

RM 求 $g(\mathbf w) = 0$；当 $g(\mathbf w) = \nabla F(\mathbf w)$ 时，RM 就退化为\textbf{SGD}（随机梯度下降）：
$$\mathbf w_{k+1} = \mathbf w_k - \alpha_k \widetilde \nabla F(\mathbf w_k, \eta_k)$$

本 notebook 在二次目标 $F(w) = \frac{1}{2}\E[(w - X)^2]$ 上验证：

1. **真梯度**：$\nabla F(w) = w - \E[X] = w - \mu^*$，零点 $w^* = \mu^*$
2. **带噪样本**：$\widetilde\nabla = w - x$，$x \sim X$（单样本估计）
3. **RM 框架对应表**：$g \leftrightarrow \nabla F$、$\eta \leftrightarrow \nabla F - \widetilde\nabla$、(C1)/(C2)/(C3) 全对应
4. **三种规模**：1-D 标量 vs 10-D 向量 vs 1000-D 大维度——看 SGD 在不同 $|\mathbf w|$ 下的行为

## 一句话

**SGD = RM with $g = \nabla F$**：所有 SGD 的收敛性自动从 RM 定理继承。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. 1-D 案例：$F(w) = \tfrac{1}{2}\E[(w-X)^2]$，$X\sim\mathcal N(5,1)$

- 真梯度 $\nabla F(w) = w - 5$
- 单样本梯度 $\widetilde\nabla = w - x$
- 噪声 $\eta = \widetilde\nabla - \nabla F = -(x - 5)$，零均值 ✓
- SGD 迭代：$w_{k+1} = w_k - \alpha_k(w_k - x_{k+1})$

**注意**：这个 SGD 迭代式与 L5 §1 的均值估计 RM 完全相同——表面看是"估均值"，本质是"SGD on quadratic"。

In [ ]:
rng = np.random.default_rng(0)
mu_star = 5.0
n_steps = 2000

def sgd_1d(alpha_fn, w0=0.0):
    w = w0
    history = [w]
    for k in range(n_steps):
        x = rng.normal(mu_star, 1.0)
        grad_sample = w - x  # 单样本梯度
        w = w - alpha_fn(k) * grad_sample
        history.append(w)
    return np.array(history)

rng = np.random.default_rng(0)
traj_decay   = sgd_1d(lambda k: 1.0/(k+1))
rng = np.random.default_rng(0)
traj_const   = sgd_1d(lambda k: 0.1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_decay, label=r'$\alpha_k = 1/(k+1)$', color='C2')
ax.plot(traj_const, label=r'$\alpha_k = 0.1$', color='C1', alpha=0.7)
ax.axhline(mu_star, color='black', linestyle='--', label='$w^*=5$')
ax.set_xlabel('iteration k'); ax.set_ylabel('w')
ax.set_title('1-D SGD on $F(w)=\\frac{1}{2}E[(w-X)^2]$ —— 与 RM 均值估计完全等价')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('figures/sgd_1d.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'1/k 衰减：w_final = {traj_decay[-1]:.4f}（误差 {abs(traj_decay[-1]-mu_star):.4f}）')
print(f'0.1 常数：w_final = {traj_const[-1]:.4f}（误差 {abs(traj_const[-1]-mu_star):.4f}）')

## 2. RM 框架对应表

| RM 元素 | 一般 RM | SGD on $F$ |
|---|---|---|
| 目标 | 求 $g(\mathbf w)=0$ | 求 $\nabla F(\mathbf w)=0$ |
| 带噪样本 | $\widetilde g(\mathbf w, \eta)$ | $\widetilde\nabla F(\mathbf w; x)$（单样本）|
| 噪声 | $\eta = \widetilde g - g$ | $\eta = \widetilde\nabla F - \nabla F$ |
| 不动点 | $\mathbf w^*$ s.t. $g(\mathbf w^*)=0$ | $\arg\min F$ |
| 步长 (C1) | $\sum\alpha=\infty, \sum\alpha^2<\infty$ | 同 |
| 噪声 (C2) | $\E[\eta]=0$ | 单样本梯度无偏 ✓ |
| Monotone (C3) | $g$ 在 $\mathbf w^*$ 附近单调 | $F$ 严格凸 ⟺ $\nabla F$ 单调 |

**关键**：SGD 收敛要求 $F$ 严格凸（或更一般的强凸 + 光滑），\textbf{这是 RM 的 (C3) 条件在凸优化语境下的具体化}。

## 3. 高维 SGD：$\mathbf w \in \R^{d}$，$F(\mathbf w) = \tfrac12\|\mathbf w - \boldsymbol\mu^*\|^2$

测试 $d=1, 10, 100, 1000$ 的收敛速度——SGD/RM 收敛与维度的关系。

In [ ]:
def sgd_nd(d, alpha_fn, n_iter=2000, seed=0):
    rng = np.random.default_rng(seed)
    mu_star_vec = rng.uniform(-3, 3, d)  # 目标向量
    w = np.zeros(d)
    err_history = [np.linalg.norm(w - mu_star_vec)]
    for k in range(n_iter):
        x = mu_star_vec + rng.standard_normal(d)
        grad_sample = w - x
        w = w - alpha_fn(k) * grad_sample
        err_history.append(np.linalg.norm(w - mu_star_vec))
    return np.array(err_history)

fig, ax = plt.subplots(figsize=(8, 4))
for d in [1, 10, 100, 1000]:
    err = sgd_nd(d, lambda k: 1.0/(k+1))
    ax.semilogy(err / np.sqrt(d), label=f'd = {d}')
ax.set_xlabel('iteration k'); ax.set_ylabel(r'$\|w_k - w^*\|_2 / \sqrt{d}$ (归一化)')
ax.set_title('SGD 收敛 vs 维度 (α=1/k)')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.savefig('figures/sgd_high_dim.png', dpi=120, bbox_inches='tight'); plt.show()

**观察**：维度归一化后（除 $\sqrt{d}$），收敛轨迹**几乎重叠**——SGD 的渐近速率与维度**无关**（只与 $\alpha_k$ 的衰减率有关）。

这是 SGD 相对二阶方法（牛顿法 $O(d^2)$）的根本优势：**适用于大模型训练**。

## 4. 课堂诊断小结

| 论断 | 数值证据 |
|---|---|
| SGD = RM with $g=\nabla F$ | 1-D 二次目标上 SGD 迭代式与均值估计 RM 完全一致 |
| $\alpha_k=1/k$ 收敛，$\alpha=$ 常数抖动 | 1-D 轨迹图直观显示 |
| SGD 渐近速率与维度无关 | $d=1, 10, 100, 1000$ 归一化曲线重叠 |
| 凸性 = RM (C3) | $F$ 严格凸 ⟺ $\nabla F$ 单调 ⟺ RM (C3) 满足 |

## 后续走向

- **L7 TD(0)**：把 $g(V) = V - T^\pi V$ 代入 RM ⇒ TD 收敛
- **L8 半梯度 TD + FA**：因为 $\widetilde\nabla$ 不再是真梯度（半梯度），RM (C3) 可能破坏 → deadly triad
- **L9 PG**：直接是 SGD on $J(\theta) = \E_\pi[G_0]$

## 思考题

1. 把 $F(w) = w^4$（非二次但仍严格凸），SGD 收敛速率是否变化？
2. 若 $F$ 非凸（如 $F(w) = w^2 - w^4$，有多个局部极小），RM 收敛保证还在吗？
3. mini-batch SGD 与 batch SGD 在 RM 框架下的区别是什么？（提示：(C2) 噪声方差）